# 08 · Make Gemma 4 Think · GRPO para razonamiento

> Entrenamos Gemma 4 (E2B o E4B, selector) a **envolver su pensamiento en `<reasoning>...</reasoning>`** antes de responder. Replica a pequeña escala el proceso que usó DeepSeek-R1.

## Qué aprendemos

1. **GRPO** (Group Relative Policy Optimization) · evolución de PPO sin reward model entrenado. Usamos **reward functions definidas en código** (ej: "premio si hay tag `<reasoning>`", "premio si la respuesta numérica es correcta").
2. **Group sampling** · para cada prompt generamos N respuestas, el reward relativo del grupo entrena al modelo.
3. **Format reward + correctness reward** · cadena de rewards apilables · primero el modelo aprende a estructurar, luego a acertar.

## Por qué importa

GRPO es el algoritmo detrás de la familia de modelos "reasoning" (DeepSeek-R1, Qwen-Reasoning, o1-style). En 2025 fue el desbloqueo que hizo viables los modelos de razonamiento open-source. Aquí lo aplicas a pequeña escala sobre Gemma 4 (E2B en T4, E4B en A100).

Es el primer notebook donde el alumno **enseña al modelo un comportamiento estructural** (no solo formato de respuesta), demostrando que el alignment va más allá de "sé buena gente".

## La cadena mental

```
PPO (06)  →  DPO (07)  →  GRPO (08 · estás aquí)
```

- PPO necesita reward model.
- DPO no necesita reward model pero sí datos de preferencia.
- GRPO no necesita reward model ni datos de preferencia · solo funciones de reward programáticas.

## Cómo funciona la reward function

```python
def format_reward(completion):
    # +1 si la respuesta contiene <reasoning>...</reasoning>
    return 1.0 if "<reasoning>" in completion and "</reasoning>" in completion else 0.0

def correctness_reward(completion, ground_truth):
    # +2 si la respuesta final es correcta
    return 2.0 if extract_answer(completion) == ground_truth else 0.0
```

El modelo aprende a maximizar la suma de rewards. Cuanto más estructurada y correcta, más reward.

## Conecta con

- **07 [ANCLA]** · DPO · alignment sobre preferencias humanas.
- **09** · evaluación: cuánto ha mejorado tu modelo (antes vs después).
- **Slides D3** · `06-grpo-reasoning` + `07-deepseek-r1-replica`.

## Stack

`transformers` + `peft` (QLoRA 4-bit) + `trl` (`GRPOTrainer`) · Gemma 4 E2B/E4B (selector) · GSM8K.

## Hardware

Selector `MODEL_SIZE`: **mini = Gemma E2B** en 4-bit -> cabe en **T4 free**; **big = E4B** -> **A100/H100**. GRPO es la técnica más pesada (genera N respuestas por prompt); si vas justo de VRAM en T4 baja `num_generations` o `max_completion_length`. **Los pasos se eligen según la GPU** (celda `GRPOConfig`): 30 en T4, 250 en A100/H100/L4. Medido en T4 (E2B, octubre 2026): ~93 s por paso de media, 30 pasos = ~46 min de entrenamiento y **~65 min el notebook entero**. En A100/H100/L4 no lo hemos cronometrado.


> **GPU**: GRPO es pesado (genera varias respuestas por prompt). Por defecto usamos **Gemma 4 E2B en 4 bits**, que cabe en una **T4**; **E4B** pide **A100**. La variante Salamandra 7B (al final) también necesita A100.


In [ ]:
from huggingface_hub import notebook_login, whoami

notebook_login()
HF_USER = whoami()["name"]
print("Conectado como:", HF_USER)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Conectado como: ericrisco


### Instalamos dependencias
Gemma 4 necesita `transformers` 5.x; `trl` trae el `GRPOTrainer`.

In [ ]:
!pip install -q "transformers>=5.4" trl peft bitsandbytes
# Colab trae un torchao antiguo que PEFT rechaza al cargar adapters ("incompatible version of torchao"); aquí no lo usamos
!pip uninstall -q -y torchao

   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.4/927.2 kB 1.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.2/927.2 kB 2.7 MB/s eta 0:00:00
   ━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/43.1 MB 18.2 MB/s eta 0:00:03

   ━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/43.1 MB 57.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━ 27.0/43.1 MB 56.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 41.0/43.1 MB 49.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 43.1/43.1 MB 55.3 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.4 MB/s eta 0:00:00


Cargamos Gemma 4 (modelo multimodal imagen-texto) desde Hugging Face, cuantizado a 4 bits para poder hacer fine-tuning en poca VRAM. Con PEFT/LoRA entrenamos solo unos adapters pequeños sobre las capas lineales del modelo de lenguaje (atención y MLP), lo que reduce mucho el coste y la memoria. También inicializamos el *processor* y su *tokenizer*.

> **NOTA:** el [blog de Unsloth](https://unsloth.ai/blog/r1-reasoning) explica bien cómo entrenar este tipo de modelos de razonamiento.


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Cambia MODEL_SIZE: "mini" = E2B (cabe en T4 free) · "big" = E4B (A100/H100)
MODEL_SIZE = "mini"
MODELS = {"mini": "google/gemma-4-E2B-it", "big": "google/gemma-4-E4B-it"}
model_name = MODELS[MODEL_SIZE]

# 4-bit (QLoRA): GRPO genera N respuestas por prompt y es pesado; en 4-bit el E2B entra en T4.
# Si vas justo de VRAM, baja num_generations o max_completion_length (más abajo).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForImageTextToText.from_pretrained(
    model_name, device_map="auto", dtype=torch.bfloat16,
    attn_implementation="eager", quantization_config=bnb_config,
)
# Ojo: NO llamamos a prepare_model_for_kbit_training. Pasa a fp32 todo lo no cuantizado, y en Gemma 4
# eso incluye los enormes embeddings por capa (PLE): pide ~9 GB más y no cabe en una T4.
# El gradient checkpointing lo activa el GRPOTrainer (GRPOConfig lo trae activado por defecto).

# LoRA solo en las capas lineales del modelo de LENGUAJE (no en las torres de visión/audio:
# aquí solo entrenamos texto, y así el adapter pesa unos MB en lugar de GB).
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

processor = AutoProcessor.from_pretrained(model_name)
tokenizer = processor.tokenizer

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Probamos el modelo antes de entrenar
Veamos cómo responde el modelo tal cual a unos problemas de aritmética, sin pedirle ningún formato.


In [ ]:
problems = [
    "Laura quiere comprar bolsas de caramelos para una fiesta. Cada bolsa cuesta 2,50 €, y quiere comprar 8 bolsas. Además, hay una oferta: si compra 10 bolsas, tiene un 20% de descuento sobre el precio total. ¿Cuánto pagará si compra solo las 8 bolsas? ¿Cuánto le costarían 10 bolsas con el descuento? ¿Le conviene comprar 8 bolsas o aprovechar la oferta y comprar 10? ¿Cuánto ahorraría o gastaría de más?",

    "El abuelo de Arnau tiene un huerto con manzanos y perales. En total tiene 45 árboles frutales. Si el número de manzanos es el doble que el de perales, ¿cuántos árboles de cada tipo tiene?",

    "Un ciclista recorre 60 km en 3 horas a velocidad constante. Si mantiene la misma velocidad, ¿cuántos kilómetros recorrerá en 5 horas?",

    "Andrea tiene 50 euros y quiere comprar 3 libros. Cada libro cuesta 12 euros. ¿Tiene dinero suficiente para comprarlos? Si no, ¿cuánto dinero le falta?"
]

### Soluciones de referencia

- **Bolsas de caramelos:** 8 bolsas cuestan 20 €; 10 bolsas con el 20% de descuento también cuestan 20 €. Le conviene comprar 10: paga lo mismo y se lleva 2 bolsas más.

- **Árboles frutales:** 15 perales y 30 manzanos.

- **Distancia del ciclista:** 100 km.

- **Andrea y los libros:** sí le alcanza (36 €). Le sobran 14 €.


In [ ]:
generation_params = {
    "temperature": 0.3,
    "top_p": 0.95,
    "max_new_tokens": 1024,
    "do_sample": True,
}

for problem in problems:
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": problem}],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt", padding=True)
    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, **generation_params)

    # Decodificamos solo los tokens nuevos (sin repetir el prompt)
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"🔹 Problema:\n\n{problem}\n\n🟦 Respuesta: {response}\n\n{'='*50}")

🔹 Problema:

Laura quiere comprar bolsas de caramelos para una fiesta. Cada bolsa cuesta 2,50 €, y quiere comprar 8 bolsas. Además, hay una oferta: si compra 10 bolsas, tiene un 20% de descuento sobre el precio total. ¿Cuánto pagará si compra solo las 8 bolsas? ¿Cuánto le costarían 10 bolsas con el descuento? ¿Le conviene comprar 8 bolsas o aprovechar la oferta y comprar 10? ¿Cuánto ahorraría o gastaría de más?

🟦 Respuesta: Aquí tienes el desglose de los cálculos y la respuesta a las preguntas:

---

### 1. Costo de 8 bolsas sin descuento

* **Precio por bolsa:** $2,50 €$
* **Cantidad de bolsas:** $8$
* **Costo total sin descuento:** $8 \times 2,50 € = 20,00 €$

**Respuesta:** Si Laura compra solo las 8 bolsas, pagará **20,00 €**.

---

### 2. Costo de 10 bolsas con el descuento

* **Precio normal por bolsa:** $2,50 €$
* **Costo total normal de 10 bolsas:** $10 \times 2,50 € = 25,00 €$
* **Descuento:** $20\%$
* **Importe del descuento:** $25,00 € \times 0,20 = 5,00 €$
* **Costo final 

🔹 Problema:

El abuelo de Arnau tiene un huerto con manzanos y perales. En total tiene 45 árboles frutales. Si el número de manzanos es el doble que el de perales, ¿cuántos árboles de cada tipo tiene?

🟦 Respuesta: Aquí está la solución al problema:

**1. Definir las variables:**

* Sea $M$ el número de manzanos.
* Sea $P$ el número de perales.

**2. Establecer las ecuaciones basadas en la información dada:**

* **Ecuación 1 (Total de árboles):** El número total de árboles es 45.
   $$M + P = 45$$
* **Ecuación 2 (Relación entre manzanos y perales):** El número de manzanos es el doble que el de perales.
   $$M = 2P$$

**3. Resolver el sistema de ecuaciones:**

Sustituimos la Ecuación 2 en la Ecuación 1:

$$(2P) + P = 45$$

Combinamos los términos de $P$:

$$3P = 45$$

Despejamos $P$ (el número de perales):

$$P = \frac{45}{3}$$
$$P = 15$$

**4. Calcular el número de manzanos:**

Usamos la Ecuación 2 para encontrar $M$:

$$M = 2P$$
$$M = 2 \times 15$$
$$M = 30$$

**5. Verificación:**

Co

🔹 Problema:

Un ciclista recorre 60 km en 3 horas a velocidad constante. Si mantiene la misma velocidad, ¿cuántos kilómetros recorrerá en 5 horas?

🟦 Respuesta: Aquí está la solución paso a paso:

**1. Calcular la velocidad del ciclista:**

La fórmula para la velocidad es:
$$\text{Velocidad} = \frac{\text{Distancia}}{\text{Tiempo}}$$

Datos iniciales:
* Distancia = 60 km
* Tiempo = 3 horas

$$\text{Velocidad} = \frac{60 \text{ km}}{3 \text{ horas}} = 20 \text{ km/h}$$

La velocidad del ciclista es de **20 kilómetros por hora**.

**2. Calcular la distancia recorrida en 5 horas:**

Ahora usamos la fórmula de la distancia:
$$\text{Distancia} = \text{Velocidad} \times \text{Tiempo}$$

Datos nuevos:
* Velocidad = 20 km/h
* Tiempo = 5 horas

$$\text{Distancia} = 20 \text{ km/h} \times 5 \text{ horas} = 100 \text{ km}$$

**Respuesta:**

El ciclista recorrerá **100 kilómetros** en 5 horas.



🔹 Problema:

Andrea tiene 50 euros y quiere comprar 3 libros. Cada libro cuesta 12 euros. ¿Tiene dinero suficiente para comprarlos? Si no, ¿cuánto dinero le falta?

🟦 Respuesta: Aquí está la solución al problema:

**1. Calcular el costo total de los libros:**

* Precio de un libro: 12 euros
* Número de libros: 3
* Costo total = $12 \text{ euros/libro} \times 3 \text{ libros} = 36 \text{ euros}$

**2. Comparar el dinero que tiene con el costo total:**

* Dinero que tiene Andrea: 50 euros
* Costo total de los libros: 36 euros

Como $50 \text{ euros} > 36 \text{ euros}$, **sí, tiene dinero suficiente para comprarlos.**

**3. Calcular cuánto dinero le sobra (opcional, pero útil):**

* Dinero restante = Dinero inicial - Costo total
* Dinero restante = $50 \text{ euros} - 36 \text{ euros} = 14 \text{ euros}$

**Respuesta:**

Sí, **tiene dinero suficiente** para comprar los libros. Le sobrarían 14 euros.



✅ **Qué observar:** compara con las soluciones de referencia. Gemma 4 ya razona paso a paso por su cuenta (cadena de pensamiento interna), pero **no** usa ningún formato fijo que podamos verificar automáticamente. Eso es justo lo que le vamos a enseñar con GRPO: separar el razonamiento (`<reasoning>`) de la respuesta final (`<answer>`), y premiar que la respuesta sea correcta.


### Preparación de los datos

Aquí verás algo curioso: ¡nuestro dataset solo contiene entradas y salidas! (en concreto, del dataset **GSM8K** de problemas de matemáticas de primaria).

Pero, un momento: ¿no decíamos que esto era distinto del **SFT**? ¡Si parece exactamente igual!

Seguimos necesitando preguntas y respuestas para verificar que el modelo aprende *algo* útil. La diferencia clave es que **no usamos un reward model de preferencias humanas ni un reward model de proceso** para inculcar las respuestas en el modelo. Solo necesitamos una forma de comprobar si una respuesta generada es correcta o no. Es decir, ¡solo necesitamos una forma de *recompensar* las respuestas correctas!

Veamos de momento qué pinta tiene el dataset de entrada.

> **NOTA:** basado en el [gist de Will Brown](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb).


In [ ]:
import re
from datasets import load_dataset, Dataset

# Cargamos y preparamos el dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# Cada ejemplo: prompt conversacional (system con el formato + pregunta) y la respuesta numérica
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Como ves, en los datos no hay ninguna información sobre preferencias, sobre "cómo razonar" ni nada parecido. Solo presentamos la pregunta y la respuesta.

Esta es la idea central de este estilo de entrenamiento: **no le decimos al modelo *cómo* debe pensar**; le dejamos explorar dentro de un entorno definido por la pregunta y la respuesta.

> **NOTA:** no es exactamente lo que hizo **DeepSeek-R1**, que usa una **pequeña** cantidad de **SFT** (el llamado "cold start") para "preparar" al modelo antes de entrar en la fase de entrenamiento por **RL**.


In [ ]:
dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'content': '\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>\n',
   'role': 'system'},
  {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
   'role': 'user'}]}

Llegamos a la *magia* de este enfoque: un conjunto de **funciones de recompensa** (reward functions).

Fíjate en que hacemos una serie de **comprobaciones** que se combinan como en este diagrama:

![image](https://i.imgur.com/7Dp0qdt.png)

En esencia, usamos varias **funciones de recompensa** para evaluar si el modelo aprende *lo que queremos*, en lugar de darle ejemplos explícitos que le dicten *cómo* debe aprender.

Estas funciones son totalmente **personalizables**: te permiten **guiar** **cómo** y **en qué** se especializa el modelo.

| Función de recompensa | Para qué sirve |
|---|---|
| `correctness_reward_func` | Premia al modelo cuando su respuesta coincide con la correcta |
| `int_reward_func` | Premia que la respuesta sea un número |
| `strict_format_reward_func` y `soft_format_reward_func` | Premian seguir el formato pedido |
| `xmlcount_reward_func` | Premia el uso correcto de las etiquetas XML y penaliza contenido extra tras la etiqueta de cierre |


In [ ]:
# Funciones de recompensa
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n?$"
    responses = [completion[0]["content"] for completion in completions]
    # re.DOTALL: sin él, "." no cruza saltos de línea y esta recompensa daría SIEMPRE 0
    # (el razonamiento ocupa varias líneas). Es un bug del gist original: revisa siempre tus rewards.
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]   # DOTALL: ver nota arriba
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

### Entrenamos el modelo

Ya tenemos:

1. **Ejemplos de entrenamiento**
2. **Funciones de recompensa**

¡Solo falta entrenar!

Empezamos fijando unos **hiperparámetros**.

> **NOTA:** están pensados para la **T4** gratuita de Colab; ajústalos a tu hardware si tienes más. El número de pasos se ajusta solo según la GPU (ver la celda siguiente).


### GRPOConfig

Primero, los **hiperparámetros típicos** de siempre (learning rate, optimizador, scheduler...).

Verás que casi no tocamos hiperparámetros **específicos de GRPO**: usamos los valores por defecto para que el notebook sea manejable, pero explora la documentación de **TRL** y ajústalos para tu caso. Los dos que más pesan en memoria y tiempo son `num_generations` (respuestas por prompt) y `max_completion_length` (longitud máxima de cada respuesta).

**Cuántos pasos.** En una T4 cada paso de GRPO es lento (genera 4 respuestas de hasta 768 tokens y luego hace el backward). Medido en T4: entre ~65 y ~100 s por paso según lo largas que salgan las respuestas (~93 s de media). Con 250 pasos serían varias horas, así que la celda elige `MAX_STEPS` según la GPU: **30 en T4** y **250 en A100/H100/L4** (en esas GPUs no lo hemos cronometrado). Si quieres forzar otro valor, cambia la línea `MAX_STEPS = ...`.

Con tan pocos pasos subimos un poco el learning rate (`2e-5`) y usamos **4 respuestas por prompt** (`num_generations = 4`): si las 2 respuestas de un grupo puntúan igual, la ventaja relativa es 0 y ese paso no enseña nada; con 4 es más fácil que haya diferencias.

> **NOTA:** si quieres ver la clásica gráfica de **RL** con "la línea que sube", quita `report_to = "none"` y conecta un logger (TensorBoard, W&B...). Aquí, tras entrenar, imprimimos la recompensa de cada paso.


In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_completion_length = 768   # tokens máximos de cada respuesta generada

# Pasos según la GPU: en T4 cada paso tarda más de un minuto, así que hacemos pocos
GPU = torch.cuda.get_device_name(0)
GPU_GRANDE = any(g in GPU for g in ("A100", "H100", "L4"))
MAX_STEPS = 250 if GPU_GRANDE else 30
# MAX_STEPS = 100   # <- descomenta (y pon tu número) para forzar otro valor
print(f"GPU: {GPU} -> MAX_STEPS = {MAX_STEPS}")


training_args = GRPOConfig(
    learning_rate = 2e-5,        # algo más alto de lo habitual (5e-6) porque hacemos pocos pasos
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_steps = 0.1,          # en transformers 5.x, un float < 1 es una fracción de los pasos (antes warmup_ratio)
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,   # 2 x 2 = 4 respuestas por paso: un grupo de 4 para cada prompt
    num_generations = 4,
    max_completion_length = max_completion_length,
    num_train_epochs = 1,
    max_steps = MAX_STEPS,
    save_steps = MAX_STEPS,
    max_grad_norm = 0.1,
    report_to = "none",
)

GPU: Tesla T4 -> MAX_STEPS = 30


Por fin, ¡lanzamos el **trainer**!

La idea de este enfoque basado en **RL** es que, en vez de mirar cómo baja la loss, miramos cómo **sube la recompensa** (la imprimimos paso a paso justo después de entrenar). En GRPO la loss ronda 0 casi todo el rato: no es un error, es cómo está definida.

> **NOTA:** en ejecuciones largas se describe un momento "¡Ajá!" en el que la recompensa pasa de ~0 a subir de golpe. Gemma 4 ya sigue bastante bien el formato desde el principio, así que lo normal es ver el `reward` alto desde los primeros pasos y que lo que mejore sea sobre todo `correctness_reward_func`.


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = processor,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

-------------------- Question:
Ahmed and Emily are having a contest to see who can get the best grade in the class. There have been 9 assignments and Ahmed has a 91 in the class. Emily has a 92. The final assignment is worth the same amount as all the other assignments. Emily got a 90 on the final assignment. What is the minimum grade Ahmed needs to get to beat Emily if all grades are whole numbers? 
Answer:
100 
Response:
<reasoning>
Let $N$ be the total number of assignments, so $N=9$.
Let $G_A$ be the current total score for Ahmed, and $G_E$ be the current total score for Emily, assuming we only consider the first 8 assignments. Since the problem doesn't give the scores for the first 8 assignments, we must infer a way to set up the problem based on the information provided.

The problem states:
1. There are 9 assignments.
2. Ahmed has a 91 in the class (this likely means Ahmed's current grade is 91 based on the first 8 assignments, or his average grade so far).
3. Emily has a 92 (Em

Step,Training Loss
1,0.000000
2,0.015907
3,0.000000
4,0.000000
5,0.000000
6,0.180542
7,0.045135
8,0.000000
9,0.216728
10,-0.042776


-------------------- Question:
The gauge on a water tank shows that the tank is 1/3 full of water. To fill the tank, 16 gallons of water are added. How many gallons of water does the tank hold when full? 
Answer:
24 
Response:
<reasoning>
Let $V$ be the total volume of the tank in gallons when full.
The gauge shows the tank is 1/3 full, so the current amount of water is $\frac{1}{3}V$.
When 16 gallons of water are added, the tank is full, so the total amount of water is $V$.
The relationship can be expressed as:
Current volume + Added volume = Total volume
$\frac{1}{3}V + 16 = V$

Now, solve for $V$:
Subtract $\frac{1}{3}V$ from both sides:
$16 = V - \frac{1}{3}V$
$16 = \left(1 - \frac{1}{3}\right)V$
$16 = \frac{2}{3}V$

To find $V$, multiply both sides by $\frac{3}{2}$:
$V = 16 \times \frac{3}{2}$
$V = \frac{16 \times 3}{2} = \frac{48}{2} = 24$

Alternatively, if the tank is 1/3 full, then the remaining empty space is $1 - \frac{1}{3} = \frac{2}{3}$ of the tank.
So, $\frac{2}{3}V = 16

-------------------- Question:
There are 15 tables in the school's cafeteria. Each table can seat 10 people. Usually, only 1/10 of the seats are left unseated. How many seats are usually taken? 
Answer:
135 
Response:
<reasoning>
The problem asks for the number of seats usually taken in the cafeteria.
1. Identify the number of tables: 15.
2. Identify the seating capacity per table: 10 people.
3. Calculate the total seating capacity: Total tables * Seats per table = $15 \times 10 = 150$ seats.
4. Determine the fraction of seats that are left unseated: 1/10 of the seats are left unseated.
5. Calculate the fraction of seats that are taken: If 1/10 are unseated, then $1 - 1/10 = 9/10$ of the seats are taken.
6. Calculate the number of seats usually taken: (Fraction taken) * (Total seating capacity) = $(9/10) \times 150$.
$ (9/10) \times 150 = 9 \times (150 / 10) = 9 \times 15 = 135$.
Alternatively, calculate the number of unseated seats first: $1/10 \times 150 = 15$. Then, the number of ta

-------------------- Question:
Fiona completed 36 math questions in an hour. Shirley was able to complete twice as many math questions within that same time, and Kiana completed half of the sum of Fiona and Shirley's math questions. If they each did the same number of questions the following hour, how many math questions did all three girls complete in 2 hours? 
Answer:
324 
Response:
<reasoning>
1. Determine the number of questions Fiona completed in one hour: Fiona = 36 questions/hour.
2. Determine the number of questions Shirley completed in one hour: Shirley completed twice as many as Fiona, so Shirley = 2 * 36 = 72 questions/hour.
3. Determine the number of questions Kiana completed in one hour: Kiana completed half the sum of Fiona and Shirley's questions.
   Sum of Fiona and Shirley = 36 + 72 = 108.
   Kiana = 108 / 2 = 54 questions/hour.
4. Determine the number of questions each person completes in the following hour: They each did the same number of questions the following hou

...

-------------------- Question:
At Mario's barbershop haircuts are 50% more expensive during the weekends. If Mario paid $18 for his last haircut on Monday, how much he would have paid the day before? 
Answer:
27 
Response:
<reasoning>
Let $P$ be the regular price of a haircut.
The price during the weekend is 50% more expensive than the regular price.
Weekend price = $P + 0.50P = 1.5P$.

The last haircut was on Monday. Since Monday is a weekday, the price paid was the regular price, $P$.
Given that Mario paid $18 for his last haircut on Monday, so $P = 18$.

The question asks for the price he would have paid the day before, which is Sunday (since Monday was the last haircut).
Since the price increases only during the weekend, Sunday is a weekend day, so the price on Sunday would be the weekend price: $1.5P$.

Calculate the weekend price: $1.5 \times \$18$.
$1.5 \times 18 = 18 + 9 = 27$.
</reasoning>
<answer>
$27
</answer> 
Extracted:
$27


-------------------- Question:
Janet has a business selling custom collars for dogs and cats. If it takes 18 inches of nylon to make a dog collar and 10 inches to make a cat collar, how much nylon does she need to make 9 dog collars and 3 cat collars? 
Answer:
192 
Response:
<reasoning>
The problem asks for the total amount of nylon needed to make a specific number of dog collars and cat collars, given the material required for each type of collar.

1.  **Determine the nylon needed per dog collar:** 18 inches.
2.  **Determine the nylon needed per cat collar:** 10 inches.
3.  **Calculate the total nylon needed for dog collars:** Number of dog collars $\times$ Nylon per dog collar = $9 \times 18$ inches.
4.  **Calculate the total nylon needed for cat collars:** Number of cat collars $\times$ Nylon per cat collar = $3 \times 10$ inches.
5.  **Calculate the total nylon needed:** Total for dog collars + Total for cat collars.
</reasoning>
<answer>
Total nylon for dog collars: $9 \times 18$ 

-------------------- Question:
Jonsey is awake for 2/3 of the day and spends 1/2 her time awake playing outside and the rest inside. Her brother, Riley, is awake for 3/4 of the day and spends 1/3 of his day outside and the rest inside. How much time, on average, do they spend inside? 
Answer:
10 
Response:
<reasoning>
First, determine the fraction of time Jonsey spends inside.
Jonsey is awake for 2/3 of the day.
Of the time Jonsey is awake, 1/2 is spent playing outside.
The fraction of time Jonsey is outside is (2/3) * (1/2) = 2/6 = 1/3 of the day.
The fraction of time Jonsey is inside is 1 - fraction outside = 1 - 1/3 = 2/3 of the day.
Alternatively, Jonsey spends 1/2 of her awake time outside, meaning she spends 1/2 * (2/3) = 1/3 of the day outside. So, inside time is 1 - 1/3 = 2/3.

Next, determine the fraction of time Riley spends inside.
Riley is awake for 3/4 of the day.
Of the time Riley is awake, 1/3 of his time is spent outside.
The fraction of time Riley is outside is (3/4) *

-------------------- Question:
The Lion King cost 10 million to make and earned 200 million at the box office. If it earned a profit that was half of what Star Wars made and Star Wars cost 25 million to make, how many millions did Star Wars earn? 
Answer:
405 
Response:
<reasoning>
1.  **Calculate the profit for The Lion King:**
    *   Cost to make: 10 million
    *   Box office earnings: 200 million
    *   Profit = Box office earnings - Cost to make
    *   Profit (Lion King) = $200 \text{ million} - 10 \text{ million} = 190 \text{ million}$

2.  **Determine the profit for The Lion King based on the given condition:**
    *   The problem states: "If it earned a profit that was half of what Star Wars made..."
    *   Profit (Lion King) = $\frac{1}{2} \times \text{Profit (Star Wars)}$
    *   We know Profit (Lion King) = 190 million.
    *   $190 \text{ million} = \frac{1}{2} \times \text{Profit (Star Wars)}$

3.  **Calculate the profit for Star Wars:**
    *   $\text{Profit (Star War

TrainOutput(global_step=30, training_loss=0.03459208539376656, metrics={'train_runtime': 2782.3227, 'train_samples_per_second': 0.043, 'train_steps_per_second': 0.011, 'total_flos': 0.0, 'train_loss': 0.03459208539376656, 'epoch': 0.004014452027298274})

In [ ]:
# La recompensa paso a paso (con report_to="none" la barra de progreso solo enseña la loss)
import pandas as pd

hist = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
cols = ["step", "reward"] + [c for c in hist.columns if c.startswith("rewards/") and c.endswith("/mean")]
tabla = hist[cols].rename(columns=lambda c: c.replace("rewards/", "").replace("_reward_func/mean", ""))
print(tabla.round(3).to_string(index=False))

n = max(1, len(tabla) // 3)
print(f"\nRecompensa media · primeros {n} pasos: {tabla['reward'].head(n).mean():.3f} | últimos {n} pasos: {tabla['reward'].tail(n).mean():.3f}")


 step  reward  xmlcount  soft_format  strict_format   int  correctness
    1   0.125     0.125        0.000          0.000 0.000          0.0
    2   0.683     0.059        0.500          0.125 0.000          0.0
    3   3.250     0.250        0.500          0.000 0.500          2.0
    4   3.250     0.250        0.500          0.000 0.500          2.0
    5   3.250     0.250        0.500          0.000 0.500          2.0
    6   0.749    -0.251        0.500          0.500 0.000          0.0
    7   0.813    -0.187        0.500          0.500 0.000          0.0
    8   3.250     0.250        0.500          0.000 0.500          2.0
    9   0.482    -0.268        0.125          0.000 0.125          0.5
   10   3.325     0.200        0.500          0.125 0.500          2.0
   11   2.684     0.184        0.500          0.125 0.375          1.5
   12   0.697    -0.052        0.500          0.250 0.000          0.0
   13   0.596    -0.404        0.500          0.500 0.000          0.0
   14 

### Cómo leer la tabla

- Cada paso es **un solo problema** de GSM8K con 4 respuestas. Por eso `reward` salta de un paso a otro: depende sobre todo de lo difícil que sea ese problema. Un 3,25 es "las 4 respuestas aciertan y con buen formato"; un valor cerca de 0 es "ninguna acierta".
- `correctness` (hasta 2) e `int` (hasta 0,5) solo puntúan si dentro de `<answer>` hay **solo el número**. Gemma tiende a responder con una frase ("Lynne gastó 75 $"), y eso puntúa 0 aunque el cálculo sea correcto: es justo el hábito que GRPO tiene que cambiar.
- Con 30 pasos **no esperes ver la curva subir**. En nuestra validación (T4) la recompensa media fue 1,92 en los 10 primeros pasos y 1,66 en los 10 últimos: ruido, porque cada paso es un problema distinto. Para ver la tendencia hacen falta cientos de pasos (por eso en A100/H100/L4 hacemos 250).
- Si una columna se quedara siempre en 0, sospecha de la función de recompensa antes que del modelo (nos pasó con `soft_format` y `strict_format` hasta añadir `re.DOTALL`).


### Subimos el adapter a Hugging Face

In [ ]:
HF_REPO_ID = f"{HF_USER}/gemma-reasoning-grpo"   # tu usuario (del login del inicio)

# Subimos el ADAPTER (pesa poco)
model.push_to_hub(HF_REPO_ID)
processor.push_to_hub(HF_REPO_ID)
print("adapter guardado en", HF_REPO_ID)

adapter guardado en ericrisco/gemma-reasoning-grpo


### Comparamos el modelo sin entrenar con el entrenado

Cargamos Gemma 4 **una sola vez** y le aplicamos el adapter que acabamos de subir. Para obtener la respuesta del modelo **sin entrenar** desactivamos temporalmente el adapter (`disable_adapter()`); así comparamos los dos con la mitad de VRAM.

Así vemos con claridad las diferencias de comportamiento en las mismas tareas.


In [ ]:
import gc, torch
from transformers import AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Liberamos el modelo de entrenamiento antes de recargar
del trainer, model
gc.collect(); torch.cuda.empty_cache()

BASE_MODEL = model_name        # el tamaño que elegiste arriba
ADAPTER    = HF_REPO_ID        # tu adapter entrenado (del push anterior)
_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

_base = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL, device_map="auto", dtype=torch.bfloat16,
    attn_implementation="eager", quantization_config=_bnb,
)
tuned_model = PeftModel.from_pretrained(_base, ADAPTER)
tuned_model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [ ]:
problems = [
    "Una fábrica produce componentes electrónicos. El lunes produce 1.200 componentes y cada día siguiente la producción aumenta un 15%. Si sigue esta tendencia, ¿cuántos componentes producirá el viernes?",

    "Un depósito rectangular de agua se llena a razón de 25 litros por minuto. Tiene una base de 4 metros cuadrados y una altura de 3 metros. ¿Cuánto tardará en llenarse por completo si empieza vacío?",

    "Un agricultor divide su terreno en dos parcelas: una para trigo y otra para maíz. La superficie total es de 500 hectáreas. Si la parcela de trigo es el triple que la de maíz, ¿qué superficie tiene cada una?",

    "Un tren sale de la estación A a las 8:00 a 80 km/h. Otro tren sale de la estación B a las 9:30 a 120 km/h, por la misma vía pero en sentido contrario. Si las estaciones están a 500 km, ¿a qué hora se cruzarán?"
]
# Soluciones: ~2.099 componentes (1.200 × 1,15⁴) · 480 min = 8 h (12 m³ = 12.000 L) · 375 ha trigo y 125 ha maíz · a las 11:24

In [ ]:
for problem in problems:
    inputs = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ], add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    n_prompt = inputs["input_ids"].shape[1]

    with torch.no_grad():
        with tuned_model.disable_adapter():          # mismo modelo SIN el adapter = modelo base
            output_base_tokens = tuned_model.generate(**inputs, max_new_tokens=768, do_sample=False)
        output_tuned_tokens = tuned_model.generate(**inputs, max_new_tokens=768, do_sample=False)
    output_base_text = tokenizer.decode(output_base_tokens[0][n_prompt:], skip_special_tokens=True)
    output_tuned_text = tokenizer.decode(output_tuned_tokens[0][n_prompt:], skip_special_tokens=True)

    print(f"🔹 **Problema:** {problem}\n")
    print(f"🟦 **Modelo base ({BASE_MODEL}):**\n{output_base_text}\n")
    print(f"🟩 **Modelo GRPO ({ADAPTER}):**\n{output_tuned_text}\n")
    print("=" * 100, "\n")

🔹 **Problema:** Una fábrica produce componentes electrónicos. El lunes produce 1.200 componentes y cada día siguiente la producción aumenta un 15%. Si sigue esta tendencia, ¿cuántos componentes producirá el viernes?

🟦 **Modelo base (google/gemma-4-E2B-it):**
<reasoning>
El problema describe una situación de crecimiento porcentual diario, lo que implica una progresión geométrica.

Datos iniciales:
Producción del lunes ($P_L$): 1200 componentes.
Tasa de aumento diaria ($r$): 15% o 0.15.

Queremos encontrar la producción del viernes.

Día 1 (Lunes): $P_1 = 1200$
Día 2 (Martes): $P_2 = P_1 \times (1 + 0.15) = 1200 \times 1.15$
Día 3 (Miércoles): $P_3 = P_2 \times 1.15 = 1200 \times (1.15)^2$
Día 4 (Jueves): $P_4 = P_3 \times 1.15 = 1200 \times (1.15)^3$
Día 5 (Viernes): $P_5 = P_4 \times 1.15 = 1200 \times (1.15)^4$

La producción del viernes será la producción del lunes multiplicada por el factor de crecimiento (1.15) elevado al número de períodos de crecimiento entre el lunes y el viern

🔹 **Problema:** Un depósito rectangular de agua se llena a razón de 25 litros por minuto. Tiene una base de 4 metros cuadrados y una altura de 3 metros. ¿Cuánto tardará en llenarse por completo si empieza vacío?

🟦 **Modelo base (google/gemma-4-E2B-it):**
<reasoning>
Para resolver este problema, necesito seguir los siguientes pasos:
1.  **Calcular el volumen total del depósito:** El volumen de un prisma rectangular se calcula como Área de la base $\times$ Altura.
2.  **Convertir las unidades de volumen:** El volumen se calculará en metros cúbicos ($m^3$), y la tasa de llenado está en litros por minuto. Necesitaré la conversión de $1 m^3 = 1000$ litros.
3.  **Calcular el tiempo total:** El tiempo será el volumen total dividido por la tasa de llenado.

**Paso 1: Calcular el volumen.**
Base = $4 m^2$
Altura = $3 m$
Volumen ($V$) = Base $\times$ Altura = $4 m^2 \times 3 m = 12 m^3$.

**Paso 2: Convertir el volumen a litros.**
$V$ en litros = $12 m^3 \times 1000 \frac{litros}{m^3} = 12000$ 

🔹 **Problema:** Un agricultor divide su terreno en dos parcelas: una para trigo y otra para maíz. La superficie total es de 500 hectáreas. Si la parcela de trigo es el triple que la de maíz, ¿qué superficie tiene cada una?

🟦 **Modelo base (google/gemma-4-E2B-it):**
<reasoning>
Llamemos $T$ a la superficie de la parcela de trigo y $M$ a la superficie de la parcela de maíz.

Se nos dan dos condiciones:
1. La superficie total es de 500 hectáreas: $T + M = 500$
2. La parcela de trigo es el triple que la de maíz: $T = 3M$

Sustituimos la segunda ecuación en la primera:
$(3M) + M = 500$
$4M = 500$

Resolvemos para $M$:
$M = \frac{500}{4}$
$M = 125$ hectáreas.

Ahora encontramos $T$ usando $T = 3M$:
$T = 3 \times 125$
$T = 375$ hectáreas.

Verificación: $T + M = 375 + 125 = 500$. La condición se cumple.
</reasoning>
<answer>
La superficie de la parcela de maíz es de 125 hectáreas y la superficie de la parcela de trigo es de 375 hectáreas.
</answer>

🟩 **Modelo GRPO (ericrisco/gemma-reasoning

🔹 **Problema:** Un tren sale de la estación A a las 8:00 a 80 km/h. Otro tren sale de la estación B a las 9:30 a 120 km/h, por la misma vía pero en sentido contrario. Si las estaciones están a 500 km, ¿a qué hora se cruzarán?

🟦 **Modelo base (google/gemma-4-E2B-it):**
<reasoning>
1.  **Definir el problema:** Tenemos dos trenes que viajan en sentido contrario (uno de A a B, el otro de B a A) en una vía de 500 km. Necesitamos encontrar la hora de encuentro.
2.  **Datos iniciales:**
    *   Distancia total ($D$): 500 km.
    *   Tren 1 (de A a B): Sale a las 8:00, velocidad $v_1 = 80$ km/h.
    *   Tren 2 (de B a A): Sale a las 9:30, velocidad $v_2 = 120$ km/h.
3.  **Establecer un sistema de coordenadas y tiempo:**
    *   Sea $t=0$ la hora de salida del primer tren (8:00).
    *   Sea la posición de la estación A como $x=0$ y la de B como $x=500$.
    *   La ecuación de posición para el Tren 1 (sale en $t=0$): $x_1(t) = 80t$.
    *   La ecuación de posición para el Tren 2 (sale a $t = 1

### Qué observar

Compara las dos respuestas de cada problema con las soluciones de referencia:

- **Formato:** con el *system prompt* que pide `<reasoning>` y `<answer>`, Gemma 4 ya usa las etiquetas **sin entrenar**. Aquí GRPO no tiene mucho que enseñar en formato.
- **Lo que sí cambia (poco):** en nuestra validación (T4, 30 pasos), en 1 de los 4 problemas el modelo GRPO pasó a poner **solo el número** dentro de `<answer>` (`2098.8075` frente a "Producirá 2099 componentes (redondeando...)"), que es exactamente lo que premian `int_reward_func` y `correctness_reward_func`. En los otros dos que terminaron, la respuesta fue prácticamente la misma.
- **Límite de longitud:** en el problema de los trenes los dos modelos agotan los 768 tokens antes de cerrar `<answer>`. Los modelos que razonan gastan muchos tokens: `max_completion_length` y `max_new_tokens` importan.
- **La idea clave:** no le hemos dado ni un solo ejemplo de cómo razonar. Solo funciones que puntúan el resultado. Es el mismo principio que usó DeepSeek-R1 a gran escala, con miles de pasos en lugar de 30: lo que ves aquí es el mecanismo, no el resultado final.


---

## Variante opcional · Salamandra 7B (razonamiento en español)

La misma idea de GRPO se aplica a un modelo español. Si tu público es hispano o catalán, o estás
sujeto a la EU AI Act, interesa entrenar el razonamiento sobre un modelo del BSC en vez de Gemma.

- **Modelo** : `BSC-LT/salamandra-7b-instruct` (Barcelona Supercomputing Center).
- **Receta** : la misma (GRPO + reward functions), cargando con Unsloth + vLLM para acelerar.
- **Dataset** : GSM8K traducido al castellano.

GRPO no depende del idioma, solo de las funciones de recompensa: es el mismo lab con el modelo cambiado.
La implementación completa (Unsloth + vLLM) está en `notebooks.legacy/10-Salamandra 7b R1.LEGACY.ipynb`.
